# ⚽ RF-DETR (DINOv2 backbone) + ByteTrack — Soccer Multi-Object Tracking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliSheheryar/DinoV2_Meta_VLM/blob/main/notebooks/RFDETR_DINOv2_Soccer_Tracking.ipynb)

**RF-DETR** is Roboflow's real-time DETR detector; its image encoder is a **DINOv2** backbone
(DINOv2-with-registers). Here we run RF-DETR per frame, keep player/ball identities across frames
with **ByteTrack**, and render ellipses, IDs, motion traces and a ball marker.

> ⚠️ Real FIFA World Cup footage is copyrighted — use a **free / royalty-free soccer clip**
> (see `data/README.md`). Run this on a **GPU runtime** (Runtime → Change runtime type → GPU).

## 1) Setup — clone repo + install RF-DETR & Supervision

In [ ]:
!git clone -q https://github.com/AliSheheryar/DinoV2_Meta_VLM.git
%cd DinoV2_Meta_VLM
!pip install -q rfdetr supervision imageio-ffmpeg

## 2) Get a soccer clip

Pick **one**: upload your own clip, or download a free one. Set `SOURCE_VIDEO` to its path.

In [ ]:
# Option A — upload a file from your computer:
# from google.colab import files; up = files.upload(); SOURCE_VIDEO = next(iter(up))

# Option B — download a free-licensed clip by its direct URL (see data/README.md for sources):
# !python scripts/download_sample.py --url "https://.../free-soccer-clip.mp4"

SOURCE_VIDEO = "data/soccer.mp4"  # <-- set this to your clip
import os; assert os.path.exists(SOURCE_VIDEO), 'Set SOURCE_VIDEO to a real soccer clip first.'

## 3) Track — RF-DETR detects, ByteTrack keeps identities

In [ ]:
import supervision as sv
from rfdetr_tracking import SoccerTracker

tracker = SoccerTracker(model_size="base", conf=0.5)   # 'large' = more accurate, slower
TARGET_VIDEO = "soccer_tracked.mp4"

def callback(frame, index):
    annotated, _ = tracker.process_frame(frame)
    return annotated

sv.process_video(source_path=SOURCE_VIDEO, target_path=TARGET_VIDEO, callback=callback)
print('done ->', TARGET_VIDEO)

## 4) Preview the result inline

In [ ]:
import base64
from IPython.display import HTML
mp4 = open(TARGET_VIDEO, 'rb').read()
data_url = 'data:video/mp4;base64,' + base64.b64encode(mp4).decode()
HTML(f'<video width=720 controls loop autoplay><source src="{data_url}" type="video/mp4"></video>')

## 5) (Optional) Export a GIF for the README

```python
import imageio, cv2
cap = cv2.VideoCapture(TARGET_VIDEO); frames = []
while True:
    ok, f = cap.read()
    if not ok: break
    frames.append(cv2.cvtColor(cv2.resize(f, None, fx=0.5, fy=0.5), cv2.COLOR_BGR2RGB))
imageio.mimsave('assets/demo_soccer.gif', frames[::3], fps=12, loop=0)
```

**Next:** RF-DETR is COCO-pretrained (players + ball). For jersey numbers, team
assignment, or pitch keypoints, fine-tune RF-DETR on a soccer dataset — see the
[Roboflow Sports](https://github.com/roboflow/sports) repo.